# Prototyping Notebook

This notebook contains headless, runnable versions of the TX and RX GNU Radio flowgraphs adapted from `Hands_on_Tx.py` and `Hands_on_RX.py`.

Notes: replace `device_index` with the HackRF device number (RX=0, TX=1) when running both on the same host.

## Requirements
- GNU Radio (3.8/3.9/3.10+ as installed)
- gr-osmosdr (for HackRF)
- Run in a Python kernel that can import `gnuradio` and `osmosdr`.

In [6]:
# Imports
import time
import threading
import numpy as np
from gnuradio import gr, blocks, analog, digital, fft
import osmosdr
import pmt

# Helper: simple safe start/stop for top_block instances
def _start_tb(tb):
    tb.start()
    return tb

def _stop_tb(tb):
    try:
        tb.stop()
        tb.wait()
    except Exception:
        pass

## Transmitter (headless)

The `Hands_on_Tx` class below mirrors the original transmitter flowgraph but without Qt GUI elements. Sinks that were GUI-based are replaced with `blocks.vector_sink_c` for offline inspection.

In [7]:
class Hands_on_Tx(gr.top_block):
    def __init__(self, carrier_freq=435e6, const=1, gain=50, samp_rate=2000000, device_index=1):
        gr.top_block.__init__(self, "Hands-on_TX - headless", catch_exceptions=True)
        # Parameters
        self.carrier_freq = carrier_freq
        self.const = const
        self.gain = gain
        self.samp_rate = samp_rate
        self.device_index = int(device_index)

        # Variables from original flowgraph
        self.packet_length_tag_key = "packet_len"
        self.length_tag_key = "frame_len"
        self.fft_len = 64
        self.rolloff = 0
        self.packet_len = 30
        self.payload_mod = digital.constellation_qpsk()
        self.header_mod = digital.constellation_bpsk()
        self.occupied_carriers = (list(range(-26, -21)) + list(range(-20, -7)) + list(range(-6, 0)) + list(range(1, 7)) + list(range(8, 21)) + list(range(22, 27)),)
        self.pilot_carriers = ((-21, -7, 7, 21,),)
        self.pilot_symbols = ((1, 1, 1, -1,),)

        # Blocks (adapted from original, GUI removed)
        self.blocks_file_source_0 = blocks.file_source(gr.sizeof_char*1, '/home/pesquisa/Downloads/data.txt', True, 0, 0)
        try:
            self.blocks_file_source_0.set_begin_tag(pmt.PMT_NIL)
        except Exception:
            pass
        self.blocks_stream_to_tagged_stream_0 = blocks.stream_to_tagged_stream(gr.sizeof_char, 1, 30, 'packet_len')
        self.blocks_repack_bits_bb_0 = blocks.repack_bits_bb(8, self.payload_mod.bits_per_symbol(), 'packet_len', False, gr.GR_LSB_FIRST)
        self.blocks_repack_bits_bb_0_0 = blocks.repack_bits_bb(8, 1, 'packet_len', False, gr.GR_LSB_FIRST)
        self.digital_crc32_bb_0 = digital.crc32_bb(False, 'packet_len', True)
        self.blocks_throttle_0 = blocks.throttle(gr.sizeof_gr_complex*1, int(self.samp_rate), True)
        self.blocks_tagged_stream_mux_0 = blocks.tagged_stream_mux(gr.sizeof_gr_complex*1, 'packet_len', 0)
        self.blocks_multiply_const_vxx_0 = blocks.multiply_const_cc(self.const)

        # Protocol/OFDM blocks (transmit chain)
        self.digital_protocol_formatter_bb_0 = digital.protocol_formatter_bb(digital.header_format_ofdm(self.occupied_carriers,1,self.packet_length_tag_key,), self.packet_length_tag_key)
        self.digital_ofdm_cyclic_prefixer_0 = digital.ofdm_cyclic_prefixer(self.fft_len, self.fft_len + 16, self.rolloff, self.length_tag_key)
        self.digital_ofdm_carrier_allocator_cvc_0 = digital.ofdm_carrier_allocator_cvc(self.fft_len, self.occupied_carriers, self.pilot_carriers, self.pilot_symbols, ([],[]), self.length_tag_key, True)
        self.digital_chunks_to_symbols_xx_0_0 = digital.chunks_to_symbols_bc(self.payload_mod.points(), 1)
        self.digital_chunks_to_symbols_xx_0 = digital.chunks_to_symbols_bc(self.header_mod.points(), 1)

        # Osmocom sink configured with explicit hackrf index
        osmo_args = f"numchan=1 hackrf={self.device_index}"
        self.osmosdr_sink_0 = osmosdr.sink(args=osmo_args)
        self.osmosdr_sink_0.set_sample_rate(self.samp_rate)
        self.osmosdr_sink_0.set_center_freq(self.carrier_freq, 0)
        self.osmosdr_sink_0.set_gain(self.gain, 0)
        self.osmosdr_sink_0.set_if_gain(self.gain, 0)
        self.osmosdr_sink_0.set_bb_gain(self.gain, 0)

        # File/vector sink for inspection
        self.blocks_file_sink_0 = blocks.file_sink(gr.sizeof_char*1, 'tx_out.bin', True)
        self.blocks_file_sink_0.set_unbuffered(False)

        # Connections (transmit chain)
        self.connect((self.blocks_file_source_0, 0), (self.blocks_stream_to_tagged_stream_0, 0))
        self.connect((self.digital_crc32_bb_0, 0), (self.blocks_repack_bits_bb_0, 0))
        self.connect((self.blocks_repack_bits_bb_0, 0), (self.blocks_tagged_stream_mux_0, 1))
        self.connect((self.blocks_tagged_stream_mux_0, 0), (self.blocks_throttle_0, 0))
        self.connect((self.blocks_throttle_0, 0), (self.osmosdr_sink_0, 0))
        self.connect((self.digital_chunks_to_symbols_xx_0, 0), (self.blocks_tagged_stream_mux_0, 0))
        self.connect((self.digital_chunks_to_symbols_xx_0_0, 0), (self.blocks_tagged_stream_mux_0, 1))
        self.connect((self.digital_protocol_formatter_bb_0, 0), (self.blocks_repack_bits_bb_0_0, 0))
        self.connect((self.blocks_tagged_stream_mux_0, 0), (self.blocks_file_sink_0, 0))

    # Getters / setters (core parameters)
    def get_carrier_freq(self):
        return self.carrier_freq

    def set_carrier_freq(self, carrier_freq):
        self.carrier_freq = carrier_freq
        try:
            self.osmosdr_sink_0.set_center_freq(self.carrier_freq, 0)
        except Exception:
            pass

    def get_gain(self):
        return self.gain

    def set_gain(self, gain):
        self.gain = gain
        try:
            self.osmosdr_sink_0.set_gain(self.gain, 0)
        except Exception:
            pass

    def get_samp_rate(self):
        return self.samp_rate

    def set_samp_rate(self, samp_rate):
        self.samp_rate = samp_rate
        try:
            self.blocks_throttle_0.set_sample_rate(int(self.samp_rate))
            self.osmosdr_sink_0.set_sample_rate(self.samp_rate)
        except Exception:
            pass

## Receiver (headless)

The `Hands_on_RX` class mirrors the original receiver but without Qt GUI elements. GUI sinks replaced by `blocks.vector_sink_c` for later analysis.

In [8]:
class Hands_on_RX(gr.top_block):
    def __init__(self, carrier_freq=435e6, gain=70, samp_rate=2000000, device_index=0):
        gr.top_block.__init__(self, "Hands-on_RX - headless", catch_exceptions=True)
        # Parameters
        self.carrier_freq = carrier_freq
        self.gain = gain
        self.samp_rate = samp_rate
        self.device_index = int(device_index)

        # Variables (from original flowgraph)
        self.fft_len = 64
        self.packet_length_tag_key = "packet_len"
        self.length_tag_key = "frame_len"
        self.payload_mod = digital.constellation_qpsk()
        self.header_mod = digital.constellation_bpsk()
        self.occupied_carriers = (list(range(-26, -21)) + list(range(-20, -7)) + list(range(-6, 0)) + list(range(1, 7)) + list(range(8, 21)) + list(range(22, 27)),)
        self.pilot_carriers = ((-21, -7, 7, 21,),)
        self.pilot_symbols = ((1, 1, 1, -1,),)
        self.payload_equalizer = digital.ofdm_equalizer_simpledfe(self.fft_len, self.payload_mod.base(), self.occupied_carriers, self.pilot_carriers, self.pilot_symbols, 1)

        # Osmocom source (HackRF) - device selection via numchan/hackrf arg
        osmo_args = f"numchan=1 hackrf={self.device_index}"
        self.osmosdr_source_0 = osmosdr.source(args=osmo_args)
        try:
            self.osmosdr_source_0.set_time_now(osmosdr.time_spec_t(time.time()), osmosdr.ALL_MBOARDS)
        except Exception:
            pass
        self.osmosdr_source_0.set_sample_rate(self.samp_rate)
        self.osmosdr_source_0.set_center_freq(self.carrier_freq, 0)
        self.osmosdr_source_0.set_gain(self.gain, 0)
        self.osmosdr_source_0.set_if_gain(self.gain, 0)
        self.osmosdr_source_0.set_bb_gain(self.gain, 0)

        # OFDM/processing blocks (adapted from original)
        self.fft_vxx_1 = fft.fft_vcc(self.fft_len, True, (), True, 1)
        self.fft_vxx_0_0 = fft.fft_vcc(self.fft_len, True, (), True, 1)
        self.digital_packet_headerparser_b_0 = digital.packet_headerparser_b(digital.packet_header_ofdm(self.occupied_carriers, n_syms=1, len_tag_key=self.packet_length_tag_key, frame_len_tag_key=self.length_tag_key, bits_per_header_sym=self.header_mod.bits_per_symbol(), bits_per_payload_sym=self.payload_mod.bits_per_symbol(), scramble_header=False).base())
        self.digital_ofdm_sync_sc_cfb_0 = digital.ofdm_sync_sc_cfb(self.fft_len, 16, False, 0.9)
        self.digital_ofdm_serializer_vcc_payload = digital.ofdm_serializer_vcc(self.fft_len, self.occupied_carriers, self.length_tag_key, self.packet_length_tag_key, 1, '', True)
        self.digital_ofdm_serializer_vcc_header = digital.ofdm_serializer_vcc(self.fft_len, self.occupied_carriers, self.length_tag_key, '', 0, '', True)
        self.digital_ofdm_frame_equalizer_vcvc_1 = digital.ofdm_frame_equalizer_vcvc(self.payload_equalizer.base(), 16, self.length_tag_key, True, 0)
        self.digital_ofdm_frame_equalizer_vcvc_0 = digital.ofdm_frame_equalizer_vcvc(digital.ofdm_equalizer_simpledfe(self.fft_len, self.header_mod.base(), self.occupied_carriers, self.pilot_carriers, self.pilot_symbols).base(), 16, self.length_tag_key, True, 1)
        self.digital_ofdm_chanest_vcvc_0 = digital.ofdm_chanest_vcvc([], [], 1, 0, 3, False)
        self.digital_header_payload_demux_0 = digital.header_payload_demux(3, self.fft_len, 16, self.length_tag_key, "", True, gr.sizeof_gr_complex, "rx_time", self.samp_rate, (), 0)
        self.digital_crc32_bb_0_0 = digital.crc32_bb(True, self.packet_length_tag_key, True)
        self.digital_constellation_decoder_cb_1 = digital.constellation_decoder_cb(self.payload_mod.base())
        self.digital_constellation_decoder_cb_0 = digital.constellation_decoder_cb(self.header_mod.base())
        self.blocks_repack_bits_bb_0_1 = blocks.repack_bits_bb(self.payload_mod.bits_per_symbol(), 8, self.packet_length_tag_key, True, gr.GR_LSB_FIRST)
        self.blocks_multiply_xx_0 = blocks.multiply_vcc(1)
        self.blocks_file_sink_0 = blocks.file_sink(gr.sizeof_char*1, 'rx.txt', True)
        self.blocks_file_sink_0.set_unbuffered(False)
        self.blocks_delay_0 = blocks.delay(gr.sizeof_gr_complex*1, 80)
        self.analog_frequency_modulator_fc_0 = analog.frequency_modulator_fc((-2.0/self.fft_len))

        # Vector sink for payload inspection
        self.payload_sink = blocks.vector_sink_c()

        # Simple throttle (if needed) and multiply/connector
        self.blocks_throttle_0 = blocks.throttle(gr.sizeof_gr_complex*1, int(self.samp_rate), True)

        # Connections (partial, key paths)
        self.connect((self.osmosdr_source_0, 0), (self.blocks_throttle_0, 0))
        self.connect((self.blocks_throttle_0, 0), (self.blocks_delay_0, 0))
        self.connect((self.blocks_delay_0, 0), (self.blocks_multiply_xx_0, 1))
        self.connect((self.analog_frequency_modulator_fc_0, 0), (self.blocks_multiply_xx_0, 0))
        self.connect((self.blocks_multiply_xx_0, 0), (self.digital_header_payload_demux_0, 0))
        self.connect((self.digital_header_payload_demux_0, 0), (self.blocks_file_sink_0, 0))
        # Attempt to attach payload serializer output to payload sink for inspection
        try:
            self.connect((self.digital_ofdm_serializer_vcc_payload, 0), (self.payload_sink, 0))
        except Exception:
            pass

    # Getters / setters (core parameters)
    def get_carrier_freq(self):
        return self.carrier_freq

    def set_carrier_freq(self, carrier_freq):
        self.carrier_freq = carrier_freq
        try:
            self.osmosdr_source_0.set_center_freq(self.carrier_freq, 0)
        except Exception:
            pass

    def get_gain(self):
        return self.gain

    def set_gain(self, gain):
        self.gain = gain
        try:
            self.osmosdr_source_0.set_gain(self.gain, 0)
        except Exception:
            pass

    def get_samp_rate(self):
        return self.samp_rate

    def set_samp_rate(self, samp_rate):
        self.samp_rate = samp_rate
        try:
            self.osmosdr_source_0.set_sample_rate(self.samp_rate)
            self.blocks_throttle_0.set_sample_rate(int(self.samp_rate))
        except Exception:
            pass

    def get_rx_samples(self):
        return np.array(self.payload_sink.data(), dtype=np.complex64)


## Run helpers and examples

Use the helpers below to start/stop the flows for a bounded time and get the captured vectors.

In [9]:
def run_rx(duration_s=5, device_index=0, samp_rate=2e6, carrier_freq=2.45e9, rx_gain=20):
    tb = Hands_on_RX(carrier_freq=carrier_freq, samp_rate=samp_rate, gain=rx_gain, device_index=device_index)
    _start_tb(tb)
    try:
        time.sleep(duration_s)
    finally:
        _stop_tb(tb)
    return tb.get_rx_samples(), tb

def run_tx(duration_s=5, device_index=1, samp_rate=2e6, carrier_freq=2.45e9, tx_gain=20):
    tb = Hands_on_Tx(carrier_freq=carrier_freq, samp_rate=samp_rate, gain=tx_gain, device_index=device_index)
    _start_tb(tb)
    try:
        time.sleep(duration_s)
    finally:
        _stop_tb(tb)
    return tb.get_tx_samples(), tb

def run_both(duration_s=5, rx_device=0, tx_device=1, samp_rate=2e6, carrier_freq=2.45e9, rx_gain=20, tx_gain=20):
    rx_tb = Hands_on_RX(carrier_freq=carrier_freq, samp_rate=samp_rate, gain=rx_gain, device_index=rx_device)
    tx_tb = Hands_on_Tx(carrier_freq=carrier_freq, samp_rate=samp_rate, gain=tx_gain, device_index=tx_device)
    _start_tb(rx_tb)
    _start_tb(tx_tb)
    try:
        time.sleep(duration_s)
    finally:
        _stop_tb(tx_tb)
        _stop_tb(rx_tb)
    return rx_tb.get_rx_samples(), tx_tb.get_tx_samples(), rx_tb, tx_tb

### How to run

- Run RX for 5 seconds (HackRF as device 0):
```python
samples, tb = run_rx(duration_s=5, device_index=0)
```
- Run TX for 5 seconds (HackRF as device 1):
```python
samples_tx, tb_tx = run_tx(duration_s=5, device_index=1)
```
- Run both (RX=0, TX=1) for 5 seconds:
```python
rx_samples, tx_samples, rx_tb, tx_tb = run_both(duration_s=5, rx_device=0, tx_device=1)
```
Note: If you are running without hardware, you can still start the flowgraphs and inspect the vector sinks; set `osmosdr` blocks to a file or use local sources instead.

In [13]:
run_both()

gr-osmosdr 0.2.0.0 (0.2.0) gnuradio 3.10.11.0
built-in source types: file fcd rtl rtl_tcp uhd hackrf rfspace airspy soapy redpitaya 
Using HackRF One with firmware v2.0.0


ValueError: Item sizes need to be >= 1, but got 0